# PathSnap dataset exploration and preprocessing

In [ ]:
import os
import re
from pathlib import Path
from zipfile import ZipFile

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

archive_path = os.environ.get("PATHSNAP_DATA_ZIP")
DATA_ZIP = (
    Path(archive_path).expanduser()
    if archive_path
    else PROJECT_ROOT / "data" / "raw" / "Student folder.zip"
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EXPECTED_TRIPS = {"train": 80, "student_test": 10}
GAP_SECONDS = 5
EARTH_RADIUS_M = 6_371_008.8

if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Put Student folder.zip at {DATA_ZIP} or set PATHSNAP_DATA_ZIP.")

## Inventory the development archive

In [ ]:
def inventory(archive):
    split_names = "|".join(EXPECTED_TRIPS)
    pattern = re.compile(
        rf"(?:[^/]+/)?(?P<split>{split_names})/"
        r"(?P<trip_id>[0-9]+)/(?P=trip_id)\.(?P<kind>track|nodes|arcs|route)"
    )
    rows = []
    for member in archive.infolist():
        match = pattern.fullmatch(member.filename)
        if match:
            rows.append({**match.groupdict(), "member": member.filename,
                         "bytes": member.file_size})
    if not rows:
        raise ValueError("No PathSnap development files found in the archive.")
    return pd.DataFrame(rows).sort_values(["split", "trip_id", "kind"])


with ZipFile(DATA_ZIP) as archive:
    files = inventory(archive)

trip_members = files.pivot(index=["split", "trip_id"], columns="kind", values="member")
trip_members = trip_members.reindex(columns=["track", "nodes", "arcs", "route"])
missing = trip_members[["track", "nodes", "arcs"]].isna().any(axis=1)
if missing.any():
    raise ValueError(f"Missing required files for {trip_members.index[missing].tolist()}")

counts = trip_members.notna().groupby(level="split").sum()
counts.insert(0, "trips", trip_members.groupby(level="split").size())
if counts["trips"].to_dict() != EXPECTED_TRIPS:
    raise ValueError(f"Unexpected trip counts: {counts['trips'].to_dict()}")
train_ids = set(trip_members.loc["train"].index)
student_test_ids = set(trip_members.loc["student_test"].index)
if train_ids & student_test_ids:
    raise ValueError("Trip IDs overlap across development partitions.")

display(counts)
display(files.groupby(["split", "kind"])["bytes"].agg(["count", "min", "median", "max"]))

## Parse and validate the road graph and reference sequence

In [ ]:
def read_columns(archive, member, columns, dtype):
    try:
        data = np.loadtxt(archive.open(member), dtype=dtype, ndmin=2)
    except ValueError as exc:
        raise ValueError(f"{member}: expected whitespace-separated {dtype} values") from exc
    if data.size == 0 or data.shape[1] != len(columns):
        raise ValueError(f"{member}: expected nonempty rows of {len(columns)} columns")
    return pd.DataFrame(data, columns=columns)


def valid_coordinates(frame):
    return frame["longitude"].between(-180, 180) & frame["latitude"].between(-90, 90)


def distance_and_bearing(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.deg2rad, (lon1, lat1, lon2, lat2))
    delta_lon, delta_lat = lon2 - lon1, lat2 - lat1
    hav = np.sin(delta_lat / 2) ** 2 + (
        np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    distance = 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))
    east = np.sin(delta_lon) * np.cos(lat2)
    north = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(delta_lon)
    bearing = np.mod(np.rad2deg(np.arctan2(east, north)), 360)
    return distance, np.where(distance > 0, bearing, np.nan)


def load_graph(archive, members):
    nodes = read_columns(archive, members.nodes, ["longitude", "latitude"], float)
    nodes.index.name = "node_id"
    if not valid_coordinates(nodes).all():
        raise ValueError(f"{members.nodes}: invalid node coordinates")

    arcs = read_columns(archive, members.arcs, ["from_node_id", "to_node_id"], np.int64)
    arcs.index.name = "arc_id"
    if not ((arcs >= 0) & (arcs < len(nodes))).all().all():
        raise ValueError(f"{members.arcs}: arc endpoint outside the node table")

    points = nodes.to_numpy()
    start = points[arcs["from_node_id"]]
    end = points[arcs["to_node_id"]]
    arcs["length_m"], arcs["bearing_deg"] = distance_and_bearing(*start.T, *end.T)
    return nodes, arcs


def audit_graph(nodes, arcs):
    src = arcs["from_node_id"].to_numpy()
    dst = arcs["to_node_id"].to_numpy()
    adjacency = csr_matrix((np.ones(len(arcs), dtype=bool), (src, dst)),
                           shape=(len(nodes), len(nodes)))
    n_components, component_id = connected_components(adjacency, directed=False)
    return {
        "nodes": len(nodes),
        "arcs": len(arcs),
        "isolated_nodes": len(nodes) - len(np.unique(np.concatenate([src, dst]))),
        "weak_components": n_components,
        "largest_component_fraction": np.bincount(component_id).max() / len(nodes),
        "duplicate_endpoint_pairs": int(arcs.duplicated(["from_node_id", "to_node_id"]).sum()),
        "zero_length_arcs": int(arcs["length_m"].eq(0).sum()),
    }


def load_reference(archive, members, arcs):
    if pd.isna(members.route):
        return None
    route = read_columns(archive, members.route, ["arc_id"], np.int64)
    route.index.name = "route_step"
    if not route["arc_id"].between(0, len(arcs) - 1).all():
        raise ValueError(f"{members.route}: reference arc ID outside the arc table")
    endpoints = arcs.loc[route["arc_id"], ["from_node_id", "to_node_id"]].to_numpy()
    route["break_before"] = np.r_[False, endpoints[:-1, 1] != endpoints[1:, 0]]
    return route

## Clean GPS tracks and derive motion features

In [ ]:
def preprocess_track(raw):
    raw_dt = raw["timestamp_s"].diff()
    valid = valid_coordinates(raw) & np.isfinite(raw["timestamp_s"])
    track = raw.loc[valid].sort_values("timestamp_s", kind="stable")
    track = track.drop_duplicates("timestamp_s", keep="first").copy()
    if track.empty:
        raise ValueError("The trip contains no usable GPS observations.")
    track.index.name = "obs_id"

    lon, lat = track["longitude"], track["latitude"]
    distance, bearing = distance_and_bearing(lon.shift(), lat.shift(), lon, lat)
    track["dt_s"] = track["timestamp_s"].diff()
    track["step_m"] = distance
    track["gap_before"] = track["dt_s"].gt(GAP_SECONDS)
    track["row_break_before"] = track.index.to_series().diff().ne(1) & track["dt_s"].notna()
    usable_motion = track["dt_s"].gt(0) & ~track["gap_before"] & ~track["row_break_before"]
    track["speed_mps"] = (track["step_m"] / track["dt_s"]).where(usable_motion)
    track["heading_deg"] = np.where(usable_motion, bearing, np.nan)

    audit = {
        "raw_gps": len(raw),
        "missing_rows": int(raw.isna().any(axis=1).sum()),
        "invalid_rows": int((~valid).sum()),
        "non_finite_rows": int((~np.isfinite(raw).all(axis=1)).sum()),
        "raw_reversals": int(raw_dt.lt(0).sum()),
        "raw_zero_steps": int(raw_dt.eq(0).sum()),
        "duplicate_timestamps_removed": int(valid.sum() - len(track)),
        "gps": len(track),
        "one_second_steps": int(track["dt_s"].eq(1).sum()),
        "irregular_steps": int(track["dt_s"].notna().sum() - track["dt_s"].eq(1).sum()),
        "gaps": int(track["gap_before"].sum()),
        "max_gap_s": track["dt_s"].max(),
        "duration_s": track["timestamp_s"].iloc[-1] - track["timestamp_s"].iloc[0],
        "within_segment_distance_m": track.loc[usable_motion, "step_m"].sum(),
        "min_lon": lon.min(), "max_lon": lon.max(),
        "min_lat": lat.min(), "max_lat": lat.max(),
    }
    return track, audit

## Run on all trips

In [ ]:
observations_by_trip = {}
trip_audits = []

with ZipFile(DATA_ZIP) as archive:
    for (split, trip_id), members in trip_members.iterrows():
        raw = read_columns(archive, members.track,
                           ["longitude", "latitude", "timestamp_s"], float)
        track, track_audit = preprocess_track(raw)
        nodes, arcs = load_graph(archive, members)
        reference = load_reference(archive, members, arcs)
        trip_audits.append({
            "split": split, "trip_id": trip_id,
            **track_audit, **audit_graph(nodes, arcs),
            "reference_steps": len(reference) if reference is not None else 0,
            "reference_breaks": int(reference["break_before"].sum()) if reference is not None else 0,
        })
        assert track.index.is_unique and track["dt_s"].dropna().gt(0).all()
        assert track.loc[track["gap_before"], ["speed_mps", "heading_deg"]].isna().all().all()
        observations_by_trip[(split, trip_id)] = track

observations = pd.concat(observations_by_trip, names=["split", "trip_id", "obs_id"])
trip_summary = pd.DataFrame(trip_audits).set_index(["split", "trip_id"])
assert observations.index.is_unique
assert observations.groupby(level=["split", "trip_id"]).size().equals(trip_summary["gps"])
assert (
    trip_summary["raw_gps"] - trip_summary["gps"]
    == trip_summary["invalid_rows"] + trip_summary["duplicate_timestamps_removed"]
).all()

totals = trip_summary.groupby(level="split").agg(
    trips=("gps", "size"), raw_gps=("raw_gps", "sum"), clean_gps=("gps", "sum"),
    missing_rows=("missing_rows", "sum"), invalid_rows=("invalid_rows", "sum"),
    raw_reversals=("raw_reversals", "sum"), raw_zero_steps=("raw_zero_steps", "sum"),
    one_second_steps=("one_second_steps", "sum"), irregular_steps=("irregular_steps", "sum"),
    gaps_over_5s=("gaps", "sum"), longest_gap_s=("max_gap_s", "max"),
    nodes=("nodes", "sum"), arcs=("arcs", "sum"), reference_breaks=("reference_breaks", "sum"),
)
totals["one_second_pct"] = (
    100 * totals["one_second_steps"] / (totals["one_second_steps"] + totals["irregular_steps"])
)
display(totals.round(2))
display(trip_summary.groupby(level="split")[
    ["gps", "duration_s", "within_segment_distance_m", "nodes", "arcs",
     "weak_components", "isolated_nodes", "largest_component_fraction"]
].median().round(3))
display(trip_summary.nlargest(5, "max_gap_s")[
    ["gps", "gaps", "max_gap_s", "min_lon", "max_lon", "min_lat", "max_lat"]
])

## Visual checks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
trip_bins = np.histogram_bin_edges(trip_summary["gps"], bins=15)
intervals = observations["dt_s"].dropna()
interval_bins = np.geomspace(intervals.min(), max(intervals.max(), 2 * intervals.min()), 35)
for split in EXPECTED_TRIPS:
    axes[0].hist(trip_summary.loc[split, "gps"], bins=trip_bins, alpha=0.65, label=split)
    gaps = observations.loc[split, "dt_s"].dropna()
    axes[1].hist(gaps, bins=interval_bins,
                 alpha=0.65, label=split)
axes[0].set(xlabel="GPS observations per trip", ylabel="Trips")
axes[1].set(xlabel="Seconds between observations", ylabel="Steps", xscale="log", yscale="log")
for ax in axes:
    ax.legend()
plt.show()

In [ ]:
training_sizes = trip_summary.loc["train", "gps"]
sample_id = (training_sizes - training_sizes.median()).abs().idxmin()
sample_track = observations.loc[("train", sample_id)]

with ZipFile(DATA_ZIP) as archive:
    sample_members = trip_members.loc[("train", sample_id)]
    sample_nodes, sample_arcs = load_graph(archive, sample_members)
    sample_route = load_reference(archive, sample_members, sample_arcs)

latlon = sample_nodes[["latitude", "longitude"]].to_numpy()
gps = sample_track[["latitude", "longitude"]].to_numpy()
lo, hi = gps.min(axis=0), gps.max(axis=0)
margin = np.maximum((hi - lo) * 0.05, 0.001)
starts = latlon[sample_arcs["from_node_id"]]
ends = latlon[sample_arcs["to_node_id"]]
near = ((np.maximum(starts, ends) >= lo - margin)
        & (np.minimum(starts, ends) <= hi + margin)).all(axis=1)

trip_map = folium.Map(location=gps.mean(axis=0).tolist(), prefer_canvas=True)
roads_layer = folium.FeatureGroup(name="Road network near the track")
road_lines = np.stack([starts[near], ends[near]], axis=1).tolist()
if road_lines:
    folium.PolyLine(road_lines, color="#64748b", weight=1, opacity=0.45).add_to(roads_layer)
roads_layer.add_to(trip_map)

if sample_route is not None:
    reference_ids = sample_route["arc_id"].to_numpy()
    reference_lines = np.stack([starts[reference_ids], ends[reference_ids]], axis=1).tolist()
    reference_layer = folium.FeatureGroup(name="Reference route")
    folium.PolyLine(reference_lines, color="#15803d", weight=4).add_to(reference_layer)
    reference_layer.add_to(trip_map)

gps_layer = folium.FeatureGroup(name="GPS observations")
for point in gps:
    folium.CircleMarker(point, radius=2, color="#1d4ed8", fill=True,
                        fill_opacity=0.8).add_to(gps_layer)
gps_layer.add_to(trip_map)
trip_map.fit_bounds([lo.tolist(), hi.tolist()])
folium.LayerControl(collapsed=False).add_to(trip_map)
display(trip_map)

## Export processed data and plots

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
observations.to_csv(
    PROCESSED_DIR / "observations.csv.gz",
    compression={"method": "gzip", "compresslevel": 6, "mtime": 0},
    lineterminator="\n",
)
trip_summary.to_csv(PROCESSED_DIR / "trip_summary.csv", lineterminator="\n")
totals.to_csv(PROCESSED_DIR / "split_summary.csv", lineterminator="\n")
files.to_csv(PROCESSED_DIR / "archive_inventory.csv", index=False, lineterminator="\n")
fig.savefig(PROCESSED_DIR / "sampling.png", dpi=150)
trip_map.save(PROCESSED_DIR / "sample_trip_map.html")
print(f"Exported {len(observations):,} observations and {len(trip_summary)} trip summaries.")